# Simulated Annealing for the TSP

Solves the berlin52 and eil101 TSPLIB instances with Simulated Annealing (swap neighborhood, geometric cooling). Each instance runs 20 independent times with a stopping rule of 500 iterations.


In [1]:
import math
import os
import random
import time

# =========================================
# 1) Read TSP file and extract coordinates
# =========================================
def read_tsp(filename):

    coords = []
    with open(filename, 'r') as f:
        start = False
        for line in f:
            # Start reading after this line
            if "NODE_COORD_SECTION" in line:
                start = True
                continue

            # Stop when reaching EOF
            if "EOF" in line:
                break

            # Extract coordinates
            if start:
                parts = line.strip().split()
                if len(parts) == 3:
                    x = float(parts[1])
                    y = float(parts[2])
                    coords.append((x, y))

    return coords


# =========================================
# 2) Distance calculation between two cities
# =========================================
def distance(city1, city2):

    return math.sqrt((city1[0] - city2[0])**2 + (city1[1] - city2[1])**2)


# =========================================
# 3) Total distance of the route
# =========================================
def total_distance(route, coords):
    """
    Calculate total travel distance of the route (closed loop)
    """
    dist = 0

    for i in range(len(route)):
        current_city = coords[route[i]]
        next_city = coords[route[(i + 1) % len(route)]]
        dist += distance(current_city, next_city)

    return dist


# =========================================
# 4) Generate initial random solution
# =========================================
def random_solution(n):

    route = list(range(n))
    random.shuffle(route)
    return route


# =========================================
# 5) Generate neighbor solution (swap two cities)
# =========================================
def neighbor(route):

    new_route = route[:]

    # Pick two random positions
    i, j = random.sample(range(len(route)), 2)

    # Swap them
    new_route[i], new_route[j] = new_route[j], new_route[i]

    return new_route


# =========================================
# 6) Simulated Annealing Algorithm
# =========================================
def simulated_annealing(coords, T=1000, cooling=0.995, iterations=500):


    n = len(coords)

    # Generate initial random solution
    current = random_solution(n)
    current_cost = total_distance(current, coords)

    # Best solution so far
    best = current[:]
    best_cost = current_cost

    # Main loop iterations
    for i in range(iterations):

        # Generate neighbor solution
        new = neighbor(current)
        new_cost = total_distance(new, coords)

        # Difference in cost
        delta = new_cost - current_cost

        # Acceptance condition:

        if delta < 0 or random.random() < math.exp(-delta / T):
            current = new
            current_cost = new_cost

        # Update best solution found
        if current_cost < best_cost:
            best = current[:]
            best_cost = current_cost

        # Decrease temperature (cooling)
        T *= cooling

    return best_cost


# =========================================
# 7) Run for test
# =========================================
if __name__ == "__main__":

    # List of datasets
    # Look for the datasets in ../data (repository layout) or the current folder (Colab upload)
    DATA_DIR = "../data" if os.path.isdir("../data") else "."
    datasets = [os.path.join(DATA_DIR, name) for name in ["berlin52.tsp", "eil101.tsp"]]

    # Loop through each dataset
    for dataset in datasets:
        print("\nRunning on:", dataset)

        # Load city coordinates from file
        coords = read_tsp(dataset)

        results = []   # Store best distance for each run
        times = []     # Store execution time for each run

        # Run 20 times
        for i in range(20):
            start = time.time()  # Start timing

            best_distance = simulated_annealing(coords)

            end = time.time()  # End timing

            results.append(best_distance)
            times.append(end - start)

        # Print all results (each run)
        print("All Results:")
        for i, r in enumerate(results):
            print(f"Run {i+1}: {r}")

        # Calculate performance metrics
        best = min(results)
        avg = sum(results) / len(results)
        avg_time = sum(times) / len(times)


        print("Best Distance:", best)
        print("Average Distance:", avg)
        print("Average Time:", avg_time)


Running on: berlin52.tsp
All Results:
Run 1: 16844.579845095934
Run 2: 18864.852153492488
Run 3: 18065.347333335998
Run 4: 16958.1775738453
Run 5: 18503.965590026917
Run 6: 15689.630495710506
Run 7: 17676.01927440338
Run 8: 15664.316344807148
Run 9: 18643.173116664257
Run 10: 18954.509131318373
Run 11: 17809.815630371526
Run 12: 18738.212079512305
Run 13: 18977.758536508325
Run 14: 18448.011330704267
Run 15: 17242.56408368613
Run 16: 17064.04450675077
Run 17: 17615.97601944278
Run 18: 18084.179304417878
Run 19: 17963.468030837612
Run 20: 20317.151473928447
Best Distance: 15664.316344807148
Average Distance: 17906.287592743018
Average Time: 0.011805558204650879

Running on: eil101.tsp
All Results:
Run 1: 3039.0649889440692
Run 2: 3035.804390039468
Run 3: 2975.3522131199484
Run 4: 3089.2898191790277
Run 5: 3036.084819617666
Run 6: 3050.2348104625316
Run 7: 2841.3963553454464
Run 8: 2976.3803870395495
Run 9: 3146.5167739257067
Run 10: 3104.7199456347817
Run 11: 3054.8571017691916
Run 12: